# NYT AI Comment Sentiment Analysis

Analyzes the ratio of positive vs. negative sentiment toward AI in NYT article comments over time.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Load sentiment data
with open("data/sentiment.json") as f:
    articles = json.load(f)

# Flatten into a DataFrame
rows = []
for article in articles:
    for comment in article["comments"]:
        if "sentiment" in comment:
            rows.append({
                "month": article["month"],
                "headline": article["headline"],
                "commentID": comment["commentID"],
                "sentiment": comment["sentiment"],
                "confidence": comment.get("confidence", "unknown"),
                "recommendations": comment.get("recommendations", 0),
            })

df = pd.DataFrame(rows)
df["month"] = pd.to_datetime(df["month"] + "-01")
print(f"Total classified comments: {len(df)}")
df["sentiment"].value_counts()

In [ ]:
# Monthly sentiment counts
monthly = df.groupby(["month", "sentiment"]).size().unstack(fill_value=0)

# Ensure all three columns exist
for col in ["positive", "negative", "neutral"]:
    if col not in monthly.columns:
        monthly[col] = 0

monthly["total"] = monthly.sum(axis=1)
monthly["pct_positive"] = monthly["positive"] / monthly["total"] * 100
monthly["pct_negative"] = monthly["negative"] / monthly["total"] * 100
monthly["pct_neutral"] = monthly["neutral"] / monthly["total"] * 100

monthly.head()

In [ ]:
# Stacked area chart: sentiment ratio over time
fig, ax = plt.subplots(figsize=(14, 5.5))

ax.stackplot(
    monthly.index,
    monthly["pct_positive"],
    monthly["pct_neutral"],
    monthly["pct_negative"],
    labels=["Positive", "Neutral", "Negative"],
    colors=["#2ecc71", "#95a5a6", "#e74c3c"],
    alpha=0.8,
)

ax.set_title("Sentiment Toward AI in NYT Comments Over Time", fontsize=14, pad=10)
ax.set_ylabel("% of comments")
ax.set_ylim(0, 100)
ax.legend(loc="upper left")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
ax.grid(True, axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
# Comment volume + sentiment lines
fig, ax1 = plt.subplots(figsize=(14, 5.5))

ax1.bar(monthly.index, monthly["total"], width=20, alpha=0.3, color="gray", label="Comment volume")
ax1.set_ylabel("Number of comments", color="gray")

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly["pct_negative"], color="#e74c3c", linewidth=2, label="% Negative")
ax2.plot(monthly.index, monthly["pct_positive"], color="#2ecc71", linewidth=2, label="% Positive")
ax2.set_ylabel("% of comments")
ax2.set_ylim(0, 100)

ax1.set_title("NYT AI Comment Volume and Sentiment Trends", fontsize=14, pad=10)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(ax1.get_xticklabels(), rotation=45, ha="right")
ax1.spines["top"].set_visible(False)
fig.tight_layout()
plt.show()